# Runbook 2.3.b: Surveillance systems (CCTV, monitoring)  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Detect Attacks · **Maps to:** 2.3.A · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** You are building the alerting logic for the IP camera network at Crestfield Analytics. Each
camera emits a stream of motion/detection events. Given the full event log (already sorted
ascending by timestamp), identify every alert condition described below and return a sorted,
de-duplicated list of alert strings.

This mirrors the lesson's monitoring concepts: active monitoring depends on software correctly
surfacing anomalies (after-hours presence, a person lingering where they shouldn't, a "ghost
motion" pattern consistent with a camera fault or someone deliberately avoiding facial capture)
so a human operator or SIEM can act on them.

**Lesson objective.** Describe the components and layout of physical surveillance systems, explain the security differences between active and passive monitoring, and analyze vulnerabilities in surveillance networks to recommend mitigations.

### What comes in
`events`: `list[dict]`, sorted ascending by `"ts"`, each dict has:
- `"camera_id"`: `str`
- `"ts"`: `int`, Unix timestamp (seconds, UTC)
- `"motion"`: `bool`
- `"person_detected"`: `bool`
- `"area"`: `str`
- `"authorized_hours"`: `tuple[int, int]`, `(start_hour, end_hour)`, both in `0..24`

### What must come out
A **sorted, de-duplicated** `list[str]` of alert strings, e.g. `"AFTER_HOURS:CAM-01"`,
`"GHOST_MOTION:CAM-02"`, `"CROWD_SERVER_ROOM"`. Empty list if no alerts.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```python
events = [
    {"camera_id": "CAM-01", "ts": 1700000000, "motion": True, "person_detected": True,
     "area": "lobby", "authorized_hours": (8, 18)},   # hour 22 -> outside (8,18)
]
solve(events)  ->  ["AFTER_HOURS:CAM-01"]

events2 = [
    {"camera_id": "CAM-02", "ts": 1700035200, "motion": True, "person_detected": False,
     "area": "hallway", "authorized_hours": (8, 18)},
    {"camera_id": "CAM-02", "ts": 1700035260, "motion": True, "person_detected": False,
     "area": "hallway", "authorized_hours": (8, 18)},
    {"camera_id": "CAM-02", "ts": 1700035320, "motion": True, "person_detected": False,
     "area": "hallway", "authorized_hours": (8, 18)},
]
solve(events2)  ->  ["GHOST_MOTION:CAM-02"]
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Timestamps within one camera's stream may be assumed to arrive in ascending order in the input;
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(events: list[dict]) -> list[str]
```

#### Input (`events`)
`list[dict]`, sorted ascending by `"ts"`, where each dict has:
- `"camera_id"`: `str`, camera identifier, e.g. `"CAM-01"`
- `"ts"`: `int`, Unix timestamp in seconds (UTC)
- `"motion"`: `bool`, whether the camera's motion sensor triggered
- `"person_detected"`: `bool`, whether analytics confirmed a person in frame
- `"area"`: `str`, the monitored area name, e.g. `"lobby"`, `"server_room"`
- `"authorized_hours"`: `tuple[int, int]`, `(start_hour, end_hour)` UTC hours during which a
  person detection at this camera is authorized; authorized iff `start <= hour < end`

#### Output (`result`)
A **sorted, de-duplicated** `list[str]` of alert strings. Empty list if no alerts.

#### Alert rules
- **AFTER_HOURS:\<camera_id\>**: flag if any event on that camera has `person_detected == True`
  and its timestamp's UTC hour (`(ts % 86400) // 3600`) falls outside `authorized_hours`.
- **GHOST_MOTION:\<camera_id\>**: flag if that camera has 3 or more consecutive events (in
  timestamp order) with `motion == True` and `person_detected == False`. Any event with
  `motion == False` or `person_detected == True` resets the streak to 0.
- **CROWD_SERVER_ROOM**: flag once (globally, not per-camera) if more than 2 events anywhere have
  `person_detected == True` and `area == "server_room"` within any 60-second sliding window.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Output always sorted ascending and de-duplicated (a `set` internally, then `sorted(...)`).
- Empty `events` input returns `[]`.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from __future__ import annotations
from collections import defaultdict


def solve(events: list[dict]) -> list[str]:
    if not events:
        return []

    alerts: set[str] = set()

    for ev in events:
        if ev["person_detected"]:
            hour = (ev["ts"] % 86400) // 3600
            start, end = ev["authorized_hours"]
            if not (start <= hour < end):
                alerts.add(f"AFTER_HOURS:{ev['camera_id']}")

    cam_events: dict[str, list[dict]] = defaultdict(list)
    for ev in events:
        cam_events[ev["camera_id"]].append(ev)

    for cam_id, cam_evs in cam_events.items():
        streak = 0
        for ev in cam_evs:
            if ev["motion"] and not ev["person_detected"]:
                streak += 1
                if streak >= 3:
                    alerts.add(f"GHOST_MOTION:{cam_id}")
                    break
            else:
                streak = 0

    server_ts = sorted(
        ev["ts"] for ev in events
        if ev["person_detected"] and ev["area"] == "server_room"
    )
    n = len(server_ts)
    for i in range(n):
        count = 1
        for j in range(i + 1, n):
            if server_ts[j] - server_ts[i] <= 60:
                count += 1
                if count > 2:
                    alerts.add("CROWD_SERVER_ROOM")
                    break
            else:
                break
        if "CROWD_SERVER_ROOM" in alerts:
            break

    return sorted(alerts)


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

import pytest


def _ev(camera_id, ts, motion, person_detected, area, authorized_hours):
    return {
        "camera_id": camera_id,
        "ts": ts,
        "motion": motion,
        "person_detected": person_detected,
        "area": area,
        "authorized_hours": authorized_hours,
    }


def test_empty_input():
    assert impl.solve([]) == []


def test_no_alerts_daytime_person():
    events = [
        _ev("CAM-01", 1700035200, True, True, "lobby", (8, 18)),
        _ev("CAM-01", 1700057600, True, True, "lobby", (8, 18)),
    ]
    assert impl.solve(events) == []


def test_after_hours_single_camera():
    events = [
        _ev("CAM-01", 1700000000, True, True, "lobby", (8, 18)),
    ]
    assert impl.solve(events) == ["AFTER_HOURS:CAM-01"]


def test_after_hours_multiple_cameras():
    events = [
        _ev("CAM-01", 1700000000, True, True, "lobby",       (8, 18)),
        _ev("CAM-02", 1700000100, True, True, "parking",     (6, 20)),
    ]
    assert impl.solve(events) == ["AFTER_HOURS:CAM-01", "AFTER_HOURS:CAM-02"]


def test_after_hours_deduplication():
    events = [
        _ev("CAM-03", 1700000000, True, True, "lobby", (8, 18)),
        _ev("CAM-03", 1700000100, True, True, "lobby", (8, 18)),
    ]
    assert impl.solve(events) == ["AFTER_HOURS:CAM-03"]


def test_ghost_motion_exactly_three():
    events = [
        _ev("CAM-02", 1700035200, True,  False, "hallway", (8, 18)),
        _ev("CAM-02", 1700035260, True,  False, "hallway", (8, 18)),
        _ev("CAM-02", 1700035320, True,  False, "hallway", (8, 18)),
    ]
    assert impl.solve(events) == ["GHOST_MOTION:CAM-02"]


def test_ghost_motion_two_not_enough():
    events = [
        _ev("CAM-02", 1700035200, True,  False, "hallway", (8, 18)),
        _ev("CAM-02", 1700035260, True,  False, "hallway", (8, 18)),
        _ev("CAM-02", 1700035320, False, False, "hallway", (8, 18)),
    ]
    assert impl.solve(events) == []


def test_ghost_motion_streak_reset():
    events = [
        _ev("CAM-04", 1700035200, True,  False, "exit",    (8, 18)),
        _ev("CAM-04", 1700035260, True,  True,  "exit",    (8, 18)),
        _ev("CAM-04", 1700035320, True,  False, "exit",    (8, 18)),
        _ev("CAM-04", 1700035380, True,  False, "exit",    (8, 18)),
    ]
    assert impl.solve(events) == []


def test_crowd_server_room_triggers():
    events = [
        _ev("CAM-05", 1700000010, True, True, "server_room", (8, 18)),
        _ev("CAM-05", 1700000020, True, True, "server_room", (8, 18)),
        _ev("CAM-05", 1700000030, True, True, "server_room", (8, 18)),
    ]
    assert impl.solve(events) == ["AFTER_HOURS:CAM-05", "CROWD_SERVER_ROOM"]


def test_crowd_server_room_spread_out():
    events = [
        _ev("CAM-05", 1700000000, True, True, "server_room", (0, 24)),
        _ev("CAM-05", 1700000070, True, True, "server_room", (0, 24)),
        _ev("CAM-05", 1700000140, True, True, "server_room", (0, 24)),
    ]
    assert impl.solve(events) == []


def test_combined_all_three_alerts():
    events = [
        _ev("CAM-01", 1700000000, True, True, "lobby", (8, 18)),
        _ev("CAM-06", 1700035200, True, False, "stairwell", (8, 18)),
        _ev("CAM-06", 1700035260, True, False, "stairwell", (8, 18)),
        _ev("CAM-06", 1700035320, True, False, "stairwell", (8, 18)),
        _ev("CAM-07", 1700057600,    True, True, "server_room", (0, 24)),
        _ev("CAM-07", 1700057620,    True, True, "server_room", (0, 24)),
        _ev("CAM-07", 1700057640,    True, True, "server_room", (0, 24)),
    ]
    result = impl.solve(events)
    assert "AFTER_HOURS:CAM-01" in result
    assert "GHOST_MOTION:CAM-06" in result
    assert "CROWD_SERVER_ROOM" in result
    assert result == sorted(result)
    assert len(result) == len(set(result))


def test_output_is_sorted():
    events = [
        _ev("CAM-ZZ", 1700000000, True, True, "lobby",   (8, 18)),
        _ev("CAM-AA", 1700000100, True, True, "parking",  (8, 18)),
    ]
    result = impl.solve(events)
    assert result == sorted(result)
    assert result[0] == "AFTER_HOURS:CAM-AA"
    assert result[1] == "AFTER_HOURS:CAM-ZZ"


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve([]),
     []),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it